### 1. Jev 主要支持以下三种模式（Primitive 问答类型）

官方文档参考：[TypeSafe](https://docs.typesafe.ai/confidence)

| 原语类型 (Question type) | 核心目标 (Goal) | 返回字段 (Returns) | 取值范围与核心特点                                                                                       |
| :--- | :--- | :--- |:------------------------------------------------------------------------------------------------|
| **`Choice`**（单选分类模式） | 从一组**无序**候选字典中选择最匹配的一项 | `choice`, `probabilities`, `confidence` | `confidence` 归一化在 `0.0 ~ 1.0`                                                                   |
| **`Noul`**（是非二分模式） | 对一个陈述判断是否为真（Yes / No） | `noul` | 返回 `0.0 ~ 1.0` 之间“为真”的概率（无单独 `confidence`，越靠近 0 或 1 越确定，`0.5` 表示完全拿不准）                          |
| **`Score`**（量表评分模式） | 根据从低到高的**有序**等级列表（共 $n$ 档）进行打分 | `score`, `probabilities`, `confidence` | `score` **未做 0~1 归一化**，范围是 **`0.0 ~ n-1`**（等级期望值）；`confidence` 归一化在 `0.0 ~ 1.0`，会额外惩罚跨等级的“两极撕裂” |

### 2. Jev (TypeSafe SDK) 三种核心模式 API 用法示例

In [2]:
from dotenv import load_dotenv

load_dotenv()
from typesafe_sdk import TypeSafeClient, Choice, Noul, Score

with TypeSafeClient() as client:
    # 模拟一段用户客服反馈文本作为输入状态
    customer_message = (
        "我这个月的 Pro 会员被重复扣款了两次！"
        "请立刻帮我把多扣的钱退掉，这已经是本周第二次出现系统故障了，非常愤怒！"
    )

    response = client.system_one(
        state={"message": customer_message},
        questions={
            # 1. Choice 模式：分类选择题（多选一，附带各选项概率及置信度）
            "department": Choice(
                instructions="应该将此消息分派给哪个团队处理？",
                criteria={
                    "billing": "涉及扣款、退款、账单或支付异常的问题",
                    "technical": "涉及应用崩溃、功能失效或系统故障的Bug",
                    "general": "其他普通咨询或账号建议"
                }
            ),

            # 2. Noul 模式：是非判断题（返回 0~1 的布尔成立概率）
            "is_angry": Noul(
                instructions="客户在文本中表现出了明显的愤怒或情绪失控？"
            ),

            # 3. Score 模式：量表评分题（根据指定的等级从低到高计算加权得分）
            "urgency": Score(
                instructions="评估该工单的紧急程度",
                criteria=[
                    "不紧急：常规询问，不影响日常使用",
                    "一般：功能受损但有临时替代方案",
                    "紧急：核心资产受损（如重复扣款/服务完全中断），需立刻人工介入"
                ]
            ),
        },
    )

### 3. 读取和解析返回结果

#### 3.1 读取 Choice 结果与置信度计算

- **Choice 置信度公式**：
  $$\text{confidence}_{\text{choice}} = \frac{p_{\max} - \frac{1}{n}}{1 - \frac{1}{n}}$$
  假设 $n=3$ 个部门，当 `billing` 概率为 `1.0` 时，$\text{confidence} = \frac{1.0 - 1/3}{1 - 1/3} = 1.0$；若三个部门概率平摊为 `(0.33, 0.33, 0.33)`，则 `confidence = 0.0`。

In [3]:
dept_answer = response.choices["department"]
print(f"【Choice 分类模式】")
print(f"  - 最佳选择: {dept_answer.choice}")
print(f"  - 置信度 (Confidence): {dept_answer.confidence}")
print(f"  - 完整概率分布 (Probabilities): {dept_answer.probabilities}")

【Choice 分类模式】
  - 最佳选择: billing
  - 置信度 (Confidence): 1.0
  - 完整概率分布 (Probabilities): {'billing': 1.0, 'technical': 0.0, 'general': 0.0}


#### 3.2 读取 Noul 结果

- 在是非题中，概率 $p$（`noul`）本身就代表了确定性：越接近 `1.0`（确信为真）或 `0.0`（确信为假）越确定，接近 `0.5` 说明模型拿不准（若需换算成 `0~1` 置信度，公式即为 $|2p - 1|$）。

In [4]:
angry_answer = response.nouls["is_angry"]
print(f"【Noul 是非模式】")
print(f"  - 为真的概率 (Probability): {angry_answer.noul}")
# 通常通过设定阈值转为布尔动作
is_customer_angry = angry_answer.noul > 0.8
print(f"  - 阈值判定 (>0.8): {'触发情绪安抚流程' if is_customer_angry else '正常处理'}")

【Noul 是非模式】
  - 为真的概率 (Probability): 0.95
  - 阈值判定 (>0.8): 触发情绪安抚流程


 #### 3.3 读取 Score 结果（直观理解 `score` 与 `confidence` 的配合）

理解 `Score` 量表评分模式，顺着下面三步走就彻底通了：

---

##### 第一步：`score` 是怎么算出来的？（整数刻度尺 + 概率加权平均）
- **刻度从 `0` 开始，不做 `0 ~ 1` 缩放**：你传入 $n$ 个**按程度从低到高**排列的 `criteria`，底层自动将它们标为整数刻度 **`0, 1, ..., n-1`**（本例共 3 档：`0=不紧急`、`1=一般`、`2=紧急`，所以得分范围是 **`0.0 ~ 2.0`**）。
- **加权平均得出连续分数**：模型先判断当前文本落在每个刻度上的概率 $p_i$，再计算加权平均分：
  $$\text{score} = 0 \times p_0 + 1 \times p_1 + 2 \times p_2$$
  例如：概率分布为 `{0: 0.0, 1: 0.0, 2: 1.0}` 时，$\text{score} = 2.0$；若介于“一般”和“紧急”之间 `{0: 0.0, 1: 0.5, 2: 0.5}`，则 $\text{score} = 1.5$。

---

##### 第二步：有了 `score`，为什么还必须看 `confidence`？（因为“平均分”会掩盖分歧！）
`confidence`（`0.0 ~ 1.0`）衡量的是**概率在尺子上的集中程度**：概率越集中在某一个刻度上，`confidence` 越接近 `1.0`；概率在尺子上跨度越分裂，`confidence` 越接近 `0.0`。

如果只看 `score` 不看 `confidence`，就会掉进**“同分不同命”**的陷阱——看下表中的 ①、③、④ 三种情况，算出来的 `score` **全都是 `1.0` 分**，但真实含义天差地别：

| 场景类型 | 概率分布 `{0:不紧急, 1:一般, 2:紧急}` | 加权得分 `score` | 置信度 `confidence` | 直观解释                                                                                                |
| :--- | :--- | :---: | :---: |:----------------------------------------------------------------------------------------------------|
| **① 笃定命中中间档** | `{0: 0.0, 1: 1.0, 2: 0.0}` | **`1.0`** | **`1.00`（高）** | 100% 落在 1 分刻度上，模型非常确定这就是普通的一般工单。                                                                    |
| **② 相邻档位轻微纠结** | `{0: 0.0, 1: 0.5, 2: 0.5}` | **`1.5`** | **`0.25`（中低）** | 只在相邻的 1 分和 2 分之间摇摆（跨度仅 1 格），大方向确定偏紧急，尚有一定可信度。                                                       |
| **③ 完全抓瞎平摊** | `{0: 0.33, 1: 0.33, 2: 0.33}` | **`1.0`** | **`0.00`（归零）** | 信息不足完全没看懂，三档平分概率，平均值恰好凑出 `1.0`，实则不可信。                                                               |
| **④ 阴阳怪气两极撕裂** | `{0: 0.5, 1: 0.0, 2: 0.5}` | **`1.0`** | **`0.00`（归零）** | 如反讽*“你们系统可真聪明呢”*：50% 像夸奖（`0`分），50% 像暴怒投诉（`2`分），首尾两极撕裂，平均分虽被拉低到温和的 `1.0`，但 `confidence = 0` 会当场识破报警！ |

> **一句话记住差异**：在有序的尺子上，晃动的跨度越小（相邻过渡），结果越靠谱；晃动的跨度越大（两极撕裂），置信度直接归零报警。

---

##### 第三步：写业务代码时怎么把两者配合起来用？
- **`score` 负责看“程度够不够高”**，**`confidence` 负责抓“模型拿不准的疑难/反讽情况交由人工兜底”**：
  ```python
  # 1. 显性高优（score >= 1.5）：明确非常紧急，直接转人工
  # 2. 隐性风险（score >= 1.0 且 confidence < 0.4）：已有负面/紧急倾向，且语义微妙模型拿不准，同样交人工稳妥复核！
  if urgency_answer.score >= 1.5 or (urgency_answer.score >= 1.0 and urgency_answer.confidence < 0.4):
      print("触发人工优先介入！")
  ```

In [5]:
urgency_answer = response.scores["urgency"]
print(f"【Score 评分模式】")
print(f"  - 计算出的加权得分: {urgency_answer.score}")
print(f"  - 各等级的概率细分: {urgency_answer.probabilities}")
print(f"  - 置信度: {urgency_answer.confidence}")

【Score 评分模式】
  - 计算出的加权得分: 2.0
  - 各等级的概率细分: {0: 0.0, 1: 0.0, 2: 1.0}
  - 置信度: 1.0


### 4. 新增box原语类型
- 回答某个目标在图像中的什么位置

In [ ]:
! uv add    pillow

In [ ]:

from pathlib import Path
from PIL import Image as PILImage,ImageDraw
from typesafe_sdk import AsyncTypeSafeClient, Box, Image, Noul

from dotenv import load_dotenv

load_dotenv()
IMAGE_PATH = Path("1.jpg")
RESULT_PATH = Path("result_1.jpg")
async def main() -> None:

    async with AsyncTypeSafeClient(
            model="jev-1.14.0",
            headers={"TypeSafe-Danger-Zone-Images": "true"},
    ) as client:
        photo = Image(IMAGE_PATH)
        response = await client.system_one(
            state={"photo": photo},
            questions={
                "has_mug": Noul(
                    instructions="照片中有没有“TYPESAFE”陶瓷马克杯？"
                ),
                "mug_location": Box(
                    instructions="“TYPESAFE”陶瓷马克杯"
                ),
            },
        )

        has_mug_ans = response.answers["has_mug"]
        mug_box_ans = response.answers["mug_location"]

        print(f"存在概率 (noul): {has_mug_ans.noul:.4f}")

        if has_mug_ans.noul < 0.5:
            print("未检测到目标杯子。")
            return

        coords = mug_box_ans.box
        print(
            f"归一化坐标: top={coords.top:.3f}, bottom={coords.bottom:.3f}, "
            f"left={coords.left:.3f}, right={coords.right:.3f} "
            f"(置信度: {mug_box_ans.confidence:.3f})"
        )

        with PILImage.open(IMAGE_PATH) as img:
            width, height = img.size

            # 将 0~1 的归一化坐标转换回像素坐标
            left_px = int(coords.left * width)
            top_px = int(coords.top * height)
            right_px = int(coords.right * width)
            bottom_px = int(coords.bottom * height)
            print(f"像素级坐标 -> [{left_px}, {top_px}, {right_px}, {bottom_px}] (原图尺寸: {width}x{height})")
            draw = ImageDraw.Draw(img)
            draw.rectangle([left_px, top_px, right_px, bottom_px], outline="red", width=5)
            img.save(RESULT_PATH)
            print(f" 线框标注完成，结果已保存至: {RESULT_PATH.resolve()}")

await main()